# CartIQ — Phase 2: Feature Engineering
**CS Machine Learning Course Project — Group 3**

Constructs all 5 feature groups: User-Level, Product-Level, User-Product, Temporal, and Sequence features.

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

orders = pd.read_parquet('data/orders.parquet')
products = pd.read_parquet('data/products.parquet')
prior = pd.read_parquet('data/prior.parquet')
train_labels = pd.read_parquet('data/train_labels.parquet')

print(f'Prior transactions: {prior.shape[0]:,}')
print(f'Train labels:       {train_labels.shape[0]:,}')
print(f'Unique users:       {prior["user_id"].nunique():,}')
print(f'Unique products:    {prior["product_id"].nunique():,}')

## 2.1 User-Level Features

In [ ]:
user_orders = orders[orders['eval_set'] == 'prior'].groupby('user_id')

user_features = pd.DataFrame({
    'u_total_orders': user_orders['order_number'].max(),
    'u_avg_days_between': user_orders['days_since_prior_order'].mean(),
    'u_std_days_between': user_orders['days_since_prior_order'].std(),
})

# Average basket size
basket_sizes = prior.groupby(['user_id', 'order_id']).size().reset_index(name='basket_size')
u_basket = basket_sizes.groupby('user_id')['basket_size'].agg(['mean', 'std']).rename(
    columns={'mean': 'u_avg_basket_size', 'std': 'u_std_basket_size'}
)
user_features = user_features.join(u_basket)

# User reorder rate
u_reorder = prior.groupby('user_id')['reordered'].mean().rename('u_reorder_rate')
user_features = user_features.join(u_reorder)

user_features = user_features.fillna(0).reset_index()
print(f'User features: {user_features.shape}')
display(user_features.head())

## 2.2 Product-Level Features

In [ ]:
prod_group = prior.groupby('product_id')

product_features = pd.DataFrame({
    'p_total_orders': prod_group.size(),
    'p_reorder_rate': prod_group['reordered'].mean(),
    'p_avg_cart_position': prod_group['add_to_cart_order'].mean(),
    'p_std_cart_position': prod_group['add_to_cart_order'].std(),
    'p_unique_users': prod_group['user_id'].nunique(),
})

product_features = product_features.fillna(0).reset_index()
print(f'Product features: {product_features.shape}')
display(product_features.head())

## 2.3 User-Product Features

In [ ]:
up_group = prior.groupby(['user_id', 'product_id'])

up_features = pd.DataFrame({
    'up_times_ordered': up_group.size(),
    'up_times_reordered': up_group['reordered'].sum(),
    'up_first_order': up_group['order_number'].min(),
    'up_last_order': up_group['order_number'].max(),
    'up_avg_cart_position': up_group['add_to_cart_order'].mean(),
})

up_features['up_reorder_ratio'] = (
    up_features['up_times_reordered'] / up_features['up_times_ordered']
).fillna(0)

# Order streak: how many of the last N orders included this product
# We compute it by checking the last 5 orders per user
user_max_order = prior.groupby('user_id')['order_number'].max().rename('max_order')
up_features = up_features.reset_index()
up_features = up_features.merge(user_max_order, on='user_id', how='left')
up_features['up_order_span'] = up_features['max_order'] - up_features['up_first_order'] + 1
up_features['up_order_frequency'] = up_features['up_times_ordered'] / up_features['up_order_span'].clip(lower=1)
up_features['up_days_since_last'] = up_features['max_order'] - up_features['up_last_order']

up_features = up_features.drop(columns=['max_order'])
print(f'User-Product features: {up_features.shape}')
display(up_features.head())

## 2.4 Temporal Features

In [ ]:
# Get the last order info per user for temporal context
last_orders = orders[orders['eval_set'] == 'prior'].sort_values('order_number').groupby('user_id').tail(1)
temporal_features = last_orders[['user_id', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']].rename(
    columns={
        'order_dow': 't_last_order_dow',
        'order_hour_of_day': 't_last_order_hour',
        'days_since_prior_order': 't_last_days_since_prior',
    }
)

# Purchase velocity: orders per week
user_span = orders[orders['eval_set'] == 'prior'].groupby('user_id').agg(
    total_days=('days_since_prior_order', 'sum'),
    total_orders=('order_id', 'count')
)
user_span['t_purchase_velocity'] = (user_span['total_orders'] / (user_span['total_days'] / 7).clip(lower=1))
temporal_features = temporal_features.merge(user_span[['t_purchase_velocity']], on='user_id', how='left')

print(f'Temporal features: {temporal_features.shape}')
display(temporal_features.head())

## 2.5 GRU Sequence Data
Per-user product ID sequences sorted by order_number, padded/truncated to length 50.

In [ ]:
MAX_SEQ_LEN = 50

# Build ordered sequences per user: list of product_ids in chronological order
user_sequences = (
    prior.sort_values(['user_id', 'order_number', 'add_to_cart_order'])
    .groupby('user_id')['product_id']
    .apply(list)
    .reset_index()
    .rename(columns={'product_id': 'sequence'})
)

# Truncate to last MAX_SEQ_LEN items
user_sequences['sequence'] = user_sequences['sequence'].apply(lambda s: s[-MAX_SEQ_LEN:])
user_sequences['seq_len'] = user_sequences['sequence'].apply(len)

print(f'User sequences: {user_sequences.shape[0]:,}')
print(f'Avg sequence length: {user_sequences["seq_len"].mean():.1f}')
print(f'Min: {user_sequences["seq_len"].min()}, Max: {user_sequences["seq_len"].max()}')

## 2.6 Assemble Final Feature Matrix

In [ ]:
# Get all (user, product) pairs from the train set
# For each train user, we need to score ALL products they've bought before
train_users = train_labels['user_id'].unique()

# Get the (user, product) pairs that exist in prior data for train users
candidate_pairs = up_features[up_features['user_id'].isin(train_users)][['user_id', 'product_id']].copy()

# Merge label: 1 if the product appears in the user's train order, 0 otherwise
train_positive = train_labels[train_labels['reordered'] == 1][['user_id', 'product_id']].copy()
train_positive['label'] = 1

feature_matrix = candidate_pairs.merge(train_positive, on=['user_id', 'product_id'], how='left')
feature_matrix['label'] = feature_matrix['label'].fillna(0).astype(int)

# Merge all feature groups
feature_matrix = feature_matrix.merge(user_features, on='user_id', how='left')
feature_matrix = feature_matrix.merge(product_features, on='product_id', how='left')
feature_matrix = feature_matrix.merge(
    up_features.drop(columns=['up_order_span']), on=['user_id', 'product_id'], how='left'
)
feature_matrix = feature_matrix.merge(temporal_features, on='user_id', how='left')

feature_matrix = feature_matrix.fillna(0)

print(f'Final feature matrix: {feature_matrix.shape}')
print(f'Label distribution:\n{feature_matrix["label"].value_counts()}')
print(f'\nPositive rate: {feature_matrix["label"].mean():.4f}')
display(feature_matrix.head())

In [ ]:
# Feature columns (exclude IDs and label)
exclude_cols = ['user_id', 'product_id', 'label']
feature_cols = [c for c in feature_matrix.columns if c not in exclude_cols]

print(f'Feature columns ({len(feature_cols)}):')
for i, c in enumerate(feature_cols):
    print(f'  {i+1:2d}. {c}')

## 2.7 Train / Validation / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# Split users (not rows) to avoid data leakage
all_users = feature_matrix['user_id'].unique()
train_users, test_users = train_test_split(all_users, test_size=0.2, random_state=42)
train_users, val_users = train_test_split(train_users, test_size=0.125, random_state=42)  # 0.125 * 0.8 = 0.1

train_df = feature_matrix[feature_matrix['user_id'].isin(train_users)]
val_df = feature_matrix[feature_matrix['user_id'].isin(val_users)]
test_df = feature_matrix[feature_matrix['user_id'].isin(test_users)]

print(f'Train: {train_df.shape[0]:,} rows ({len(train_users):,} users)')
print(f'Val:   {val_df.shape[0]:,} rows ({len(val_users):,} users)')
print(f'Test:  {test_df.shape[0]:,} rows ({len(test_users):,} users)')
print(f'\nTrain positive rate: {train_df["label"].mean():.4f}')
print(f'Val positive rate:   {val_df["label"].mean():.4f}')
print(f'Test positive rate:  {test_df["label"].mean():.4f}')

In [ ]:
# Save everything
feature_matrix.to_parquet('data/feature_matrix.parquet', index=False)
train_df.to_parquet('data/train.parquet', index=False)
val_df.to_parquet('data/val.parquet', index=False)
test_df.to_parquet('data/test.parquet', index=False)

# Save sequences
user_sequences.to_parquet('data/user_sequences.parquet', index=False)

# Save feature column names
pd.Series(feature_cols).to_csv('data/feature_cols.csv', index=False, header=False)

print('All data saved to data/ directory.')
for f in sorted(os.listdir('data')):
    size = os.path.getsize(os.path.join('data', f)) / 1e6
    print(f'  {f:35s} {size:8.2f} MB')